# Thesis pipeline on Google Colab Pro+ — Part 8 full run

Open this notebook, choose **Runtime → Change runtime type → CPU, High-RAM**, then
**Runtime → Run all**. Nothing needs editing. It will:

1. check the runtime (CPU count, memory) and mount Google Drive;
2. clone the repository branch onto the VM's local disk (not Drive);
3. build an isolated **Python 3.11** environment with exactly the approved lock
   (`code/requirements.lock.txt`) plus the validated transitive versions
   (`code/requirements.constraints.txt`), and verify every version;
4. verify the frozen dataset's hashes (no network, no analysis);
5. restore a compatible fit cache from Drive, if an earlier attempt left one;
6. start `code/run_all.py` with **4 workers** in the background, so a browser disconnect
   does not stop it, and a backup process that copies **new** fits and outputs to Drive
   every few minutes as compressed increments;
7. stream the complete progress-monitor output here until the run ends (the same log is
   mirrored to Drive every two minutes);
8. verify the outputs against the run's own checksum inventory and the frozen data, copy
   them to Drive, and optionally push them to a dedicated GitHub branch.

**If the runtime is terminated** (Colab Pro+ sessions are time-limited), reconnect and
choose **Run all** again. Step 5 restores every fit already backed up — only fits made by
this exact code, configuration and data — and the pipeline recomputes only what is
missing. Results are identical to an uninterrupted run.

**Secrets.** To clone a private repository or push results, add a GitHub token as a Colab
secret named `GITHUB_TOKEN` (key icon in the left bar; enable notebook access). The token
is read from the secret into a process environment variable only; it is never written to
a remote URL, to `.git/config`, to a command line or to this notebook.

In [ ]:
# @title Settings (defaults run the full Part 8 baseline; no edits needed)
REPOSITORY = "charlijomedia-dotcom/academia7"  # @param {type:"string"}
BRANCH = "claude/parallel-scheduling-colab"  # @param {type:"string"}
WORKERS = 4  # @param {type:"integer"}
DRIVE_ROOT = "/content/drive/MyDrive/thesis_part8"  # @param {type:"string"}
BACKUP_MINUTES = 10  # @param {type:"integer"}
RESTORE_CACHE = True  # @param {type:"boolean"}
PUSH_RESULTS = True  # @param {type:"boolean"}
RESULTS_BRANCH_PREFIX = "part8-results"  # @param {type:"string"}
SMOKE_TEST = False  # @param {type:"boolean"}
USE_CONSTRAINTS = True  # @param {type:"boolean"}

# Advanced: where the work happens on the VM's local disk, and an optional clone source
# (empty = GitHub). Leave as they are on Colab.
WORK = "/content/work"
SOURCE_URL = ""

## 1. Runtime check, Google Drive, secret

In [ ]:
import os, sys, json, time, shutil, subprocess, importlib.util
from pathlib import Path

cpus = os.cpu_count() or 1
try:
    memory_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
except (ValueError, OSError):
    memory_gb = float("nan")
print(f"CPUs: {cpus}   memory: {memory_gb:.1f} GB   notebook Python: {sys.version.split()[0]}")
if cpus < WORKERS:
    print(f"WARNING: {cpus} CPUs for {WORKERS} workers; choose a High-RAM CPU runtime.")

try:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    try:
        GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
    except Exception:
        GITHUB_TOKEN = None
except ImportError:                       # outside Colab (used only to test this notebook)
    GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")

NAMESPACE = "smoke" if SMOKE_TEST else "baseline"
DRIVE = Path(DRIVE_ROOT) / NAMESPACE
WORK_DIR = Path(WORK)
RUN_DIR = WORK_DIR / "run"
for directory in (DRIVE, WORK_DIR, RUN_DIR):
    directory.mkdir(parents=True, exist_ok=True)
print("GitHub token:", "found in Colab secrets" if GITHUB_TOKEN else "not set (public clone, no push)")
print("Drive folder:", DRIVE)

## 2. Clone the repository onto the local disk

In [ ]:
# Credentials come from GIT_ASKPASS reading an environment variable; the URL never has them.
SECRETS = WORK_DIR / "secrets"
SECRETS.mkdir(exist_ok=True)
GIT_ENV = dict(os.environ, GIT_TERMINAL_PROMPT="0")
if GITHUB_TOKEN:
    askpass = SECRETS / "git_askpass.sh"
    askpass.write_text('#!/bin/sh\ncase "$1" in\n  Username*) echo "x-access-token" ;;\n'
                       '  *) echo "$THESIS_GIT_TOKEN" ;;\nesac\n')
    askpass.chmod(0o700)
    GIT_ENV.update(GIT_ASKPASS=str(askpass), THESIS_GIT_TOKEN=GITHUB_TOKEN)

REPO = WORK_DIR / "academia7"
url = SOURCE_URL or f"https://github.com/{REPOSITORY}.git"
assert "@" not in url.split("//", 1)[-1].split("/", 1)[0], "no credentials in URLs"
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, url, str(REPO)], check=True, env=GIT_ENV)
else:
    subprocess.run(["git", "fetch", "origin", BRANCH], cwd=REPO, check=True, env=GIT_ENV)
    subprocess.run(["git", "checkout", BRANCH], cwd=REPO, check=True)
    subprocess.run(["git", "merge", "--ff-only", f"origin/{BRANCH}"], cwd=REPO, check=True)
PROJECT = REPO / "my-thesis"
CODE = PROJECT / "code"
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True,
                        text=True, check=True).stdout.strip()
print("commit:", COMMIT)

spec = importlib.util.spec_from_file_location("thesis_colab", PROJECT / "colab" / "thesis_colab.py")
tc = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tc)

## 3. Isolated Python 3.11 environment from the approved lock

In [ ]:
VENV = WORK_DIR / "venv311"
LOCK = CODE / "requirements.lock.txt"
CONSTRAINTS = CODE / "requirements.constraints.txt" if USE_CONSTRAINTS else None
PY = VENV / "bin" / "python"
try:
    tc.verify_environment(PY, LOCK, constraints=CONSTRAINTS)
    print("existing environment verified")
except Exception:
    PY = tc.create_environment(VENV, LOCK, constraints=CONSTRAINTS)
found = tc.verify_environment(PY, LOCK, constraints=CONSTRAINTS)
print(f"Python {found['python']}; {len(found['packages'])} pinned packages verified exactly")
freeze = subprocess.run([sys.executable, "-m", "uv", "pip", "freeze", "--python", str(PY)],
                        capture_output=True, text=True).stdout
(DRIVE / f"environment_{COMMIT[:12]}.txt").write_text(freeze)

## 4. Verify the frozen dataset (seconds; no network)

In [ ]:
RUN_ENV = dict(os.environ, PYTHONUNBUFFERED="1", MPLBACKEND="Agg")
RUN_ENV.pop("THESIS_GIT_TOKEN", None)                 # the analysis never sees the token
extra = ["--smoke-test"] if SMOKE_TEST else []
check = subprocess.run([str(PY), "-u", "code/run_all.py", "--data-only", *extra], cwd=PROJECT,
                       env=RUN_ENV, capture_output=True, text=True)
print(check.stdout[-2000:], check.stderr[-2000:])
if check.returncode != 0:
    raise SystemExit("frozen-data verification failed; nothing was started")
FROZEN_SHA = tc.sha256(PROJECT / "data" / "frozen" / "thesis_input_2026-10-05.csv")
print("frozen input sha256:", FROZEN_SHA)

## 5. Restore a compatible fit cache from Drive

In [ ]:
IDENTITY = tc.cache_identity(PY, CODE, NAMESPACE)
CACHE_DIR = PROJECT / ("cache" if NAMESPACE == "baseline" else "smoke_cache") / "fits"
OUTPUT_DIR = PROJECT / ("output" if NAMESPACE == "baseline" else "smoke_output")
DRIVE_CACHE = DRIVE / "fit_cache" / IDENTITY
DRIVE_OUTPUT_PARTS = DRIVE / "output_increments" / COMMIT[:12]
print("cache identity (code + configuration + data):", IDENTITY)
if RESTORE_CACHE:
    restored = tc.IncrementalArchive(CACHE_DIR, DRIVE_CACHE).restore()
    print(f"restored {restored} cached fits" if restored else "no earlier cache for this identity")
others = [p.name for p in (DRIVE / "fit_cache").glob("*") if p.name != IDENTITY]
if others:
    print("ignored (different code, configuration or data):", ", ".join(others))

## 6. Start the run and the incremental backups

In [ ]:
CONSOLE, EXIT_FILE, PID_FILE = RUN_DIR / "console.log", RUN_DIR / "exit_code", RUN_DIR / "run.pid"
BACKUP_PID = RUN_DIR / "backup.pid"
command = [str(PY), "-u", "code/run_all.py", "--workers", str(WORKERS),
           "--log-mirror", str(DRIVE / "run_logs"), *extra]
if EXIT_FILE.exists() and EXIT_FILE.read_text().strip():
    print("this VM already finished a run (exit code", EXIT_FILE.read_text().strip(), "); not restarting")
    RUN_PID = int(PID_FILE.read_text())
else:
    RUN_PID = tc.launch_background(command, cwd=PROJECT, console=CONSOLE, exit_file=EXIT_FILE,
                                   pid_file=PID_FILE, env=RUN_ENV)
    print("run_all.py started, pid", RUN_PID)
    daemon = [sys.executable, str(PROJECT / "colab" / "thesis_colab.py"), "backup-daemon",
              "--pid", str(RUN_PID), "--interval", str(BACKUP_MINUTES * 60),
              "--pair", str(CACHE_DIR), str(DRIVE_CACHE),
              "--pair", str(OUTPUT_DIR), str(DRIVE_OUTPUT_PARTS),
              "--log", str(DRIVE / "backup.log")]
    tc.launch_background(daemon, cwd=WORK_DIR, console=RUN_DIR / "backup_console.log",
                         exit_file=RUN_DIR / "backup_exit", pid_file=BACKUP_PID)
    print(f"backups every {BACKUP_MINUTES} min to {DRIVE}")

## 7. Live progress (re-run this cell to reattach after a disconnect)

Stopping this cell stops the *watching*, not the run.

In [ ]:
STATUS = tc.watch(CONSOLE, PID_FILE, EXIT_FILE, poll=5,
                  snapshot_root=PROJECT / "run_logs" / NAMESPACE)
print("\nrun_all.py exit status:", STATUS)

## 8. Final backup and integrity verification

In [ ]:
deadline = time.time() + 3600
while BACKUP_PID.exists() and tc.process_alive(int(BACKUP_PID.read_text())) and time.time() < deadline:
    time.sleep(5)                                        # the daemon's final sync
print((DRIVE / "backup.log").read_text()[-1500:] if (DRIVE / "backup.log").exists() else "")
if STATUS != 0:
    raise SystemExit("the run did not finish successfully; see the log above. "
                     "Run all again to resume from the backed-up cache.")
REPORT = tc.verify_outputs(PROJECT, namespace=NAMESPACE)
if tc.sha256(PROJECT / "data" / "frozen" / "thesis_input_2026-10-05.csv") != FROZEN_SHA:
    REPORT["ok"] = False
    REPORT["problems"].append("frozen input changed during the run")
RUN_ID = REPORT.get("run_id") or "unknown-run"
REPORT.update(commit=COMMIT, frozen_sha256=FROZEN_SHA, cache_identity=IDENTITY)
(DRIVE / f"verification_{RUN_ID}.json").write_text(json.dumps(REPORT, indent=2))
print(json.dumps(REPORT, indent=2))
if not REPORT["ok"]:
    raise SystemExit("integrity verification failed; results were not published")
RESULTS = DRIVE / "results" / RUN_ID
if RESULTS.exists():
    shutil.rmtree(RESULTS)
shutil.copytree(OUTPUT_DIR, RESULTS / OUTPUT_DIR.name)
shutil.copytree(PROJECT / "run_logs" / NAMESPACE / RUN_ID, RESULTS / "run_logs")
print("results copied to", RESULTS)

## 9. Optional: push the verified results to their own GitHub branch

In [ ]:
if not PUSH_RESULTS:
    print("PUSH_RESULTS is off; results are on Drive")
elif SMOKE_TEST:
    print("reduced (smoke) results are never pushed: they are not thesis-facing")
elif not GITHUB_TOKEN and not SOURCE_URL:
    print("no GITHUB_TOKEN secret; results are on Drive only")
else:
    branch = f"{RESULTS_BRANCH_PREFIX}/{RUN_ID}"
    sha = tc.push_results(REPO, PROJECT, branch, GIT_ENV,
                          message=f"Part 8 results {RUN_ID} (code {COMMIT[:12]})",
                          author=("Thesis pipeline (Colab)", "thesis-pipeline@users.noreply.github.com"))
    print(f"pushed {branch} at {sha}")